In [ ]:
# ============================================================
# DATATHON 2026 ROUND 1 - MCQ NOTEBOOK (Q1 -> Q10)
# ============================================================


from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
# Put all CSV files in the same folder as this notebook,
# or change DATA_DIR to your actual folder.
DATA_DIR = Path("../../data")

def load_csv(name, parse_dates=None):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    return pd.read_csv(path, parse_dates=parse_dates)

In [3]:
# ------------------------------------------------------------
# 1) LOAD ALL TABLES
# ------------------------------------------------------------
products = load_csv("products.csv")
customers = load_csv("customers.csv", parse_dates=["signup_date"])
geography = load_csv("geography.csv")
orders = load_csv("orders.csv", parse_dates=["order_date"])
order_items = load_csv("order_items.csv")
payments = load_csv("payments.csv")
returns = load_csv("returns.csv", parse_dates=["return_date"])
sales = load_csv("sales.csv", parse_dates=["Date"])   # loaded for reference / checking only
web_traffic = load_csv("web_traffic.csv", parse_dates=["date"])

print("All required files loaded successfully.")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17180\242877795.py:25: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, parse_dates=parse_dates)


All required files loaded successfully.


In [4]:
# ------------------------------------------------------------
# 2) QUICK SANITY CHECKS
# ------------------------------------------------------------
print("\n=== QUICK SANITY CHECKS ===")
print("products rows:", len(products))
print("customers rows:", len(customers))
print("geography rows:", len(geography))
print("orders rows:", len(orders))
print("order_items rows:", len(order_items))
print("payments rows:", len(payments))
print("returns rows:", len(returns))
print("sales rows:", len(sales))
print("web_traffic rows:", len(web_traffic))

print("\n=== JOIN KEY CHECKS ===")
print("order_items.order_id not in orders:", (~order_items["order_id"].isin(orders["order_id"])).sum())
print("orders.customer_id not in customers:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("orders.zip not in geography:", (~orders["zip"].isin(geography["zip"])).sum())
print("returns.order_id not in orders:", (~returns["order_id"].isin(orders["order_id"])).sum())
print("returns.product_id not in products:", (~returns["product_id"].isin(products["product_id"])).sum())
print("payments.order_id not in orders:", (~payments["order_id"].isin(orders["order_id"])).sum())


=== QUICK SANITY CHECKS ===
products rows: 2412
customers rows: 121930
geography rows: 39948
orders rows: 646945
order_items rows: 714669
payments rows: 646945
returns rows: 39939
sales rows: 3833
web_traffic rows: 3652

=== JOIN KEY CHECKS ===
order_items.order_id not in orders: 0
orders.customer_id not in customers: 0
orders.zip not in geography: 0
returns.order_id not in orders: 0
returns.product_id not in products: 0
payments.order_id not in orders: 0


In [ ]:
# ------------------------------------------------------------
# 3) Q1
# ------------------------------------------------------------
q1_orders = (
    orders[["customer_id", "order_date"]]
    .dropna()
    .sort_values(["customer_id", "order_date"])
    .copy()
)

q1_orders["prev_order_date"] = q1_orders.groupby("customer_id")["order_date"].shift(1)
q1_orders["gap_days"] = (q1_orders["order_date"] - q1_orders["prev_order_date"]).dt.days

# only gaps from customers with >= 2 orders remain non-null
q1_gaps = q1_orders["gap_days"].dropna()
q1_median_gap = q1_gaps.median()

print("\n=== Q1 ===")
print("Median inter-order gap (days):", q1_median_gap)
print(q1_gaps.describe())

q1_options = {30: "A", 90: "B", 180: "C", 365: "D"}
q1_closest = min(q1_options, key=lambda x: abs(x - q1_median_gap))
print("Closest option:", q1_options[q1_closest], f"({q1_closest} days)")


=== Q1 ===
Median inter-order gap (days): 144.0
count    556699.000000
mean        285.592509
std         389.691558
min           0.000000
25%          46.000000
50%         144.000000
75%         357.000000
max        3785.000000
Name: gap_days, dtype: float64
Closest option: C (180 days)


In [ ]:
# ------------------------------------------------------------
# 4) Q2
# ------------------------------------------------------------

q2_products = products[["segment", "price", "cogs"]].dropna().copy()
q2_products["gross_margin"] = (q2_products["price"] - q2_products["cogs"]) / q2_products["price"]

q2_segment_margins = q2_products.groupby("segment")["gross_margin"].mean()

q2_best_segment = q2_segment_margins.idxmax()
q2_max_value = q2_segment_margins.max()
q2_options={
    "Premium": "A",
    "Performance": "B", 
    "Activewear": "C",
    "Standard": "D"
}
print("\n=== Q2 ===") 
print('The best sum of gross margin& segment :', round(q2_max_value, 5),'-' ,q2_best_segment)
q2_answer= q2_options.get(q2_best_segment)
print(f"Closest option: {q2_answer} ({q2_best_segment})")


=== Q2 ===
The best sum of gross margin& segment : 0.31344 - Standard
Closest option: D (Standard)


In [ ]:
# ------------------------------------------------------------
# 5) Q3
# ------------------------------------------------------------

streetwear_products = products[products["category"] == "Streetwear"][["product_id", "category"]]
q3_merged = returns.merge(streetwear_products, on="product_id", how="inner")
q3_reason_count=q3_merged["return_reason"].value_counts().to_frame()
q3_most_reason=q3_reason_count.idxmax() 

q3_options = {
    "defective": "A",
    "wrong_size": "B",
    "changed_mind": "C",
    "not_as_described": "D"
}
print('Thống kê lý do trả hàng :')
print( q3_reason_count)
print('Lý do phổ biến nhất: ', q3_most_reason.to_frame())


Thống kê lý do trả hàng :
                  count
return_reason          
wrong_size         7626
defective          4330
not_as_described   3854
changed_mind       3830
late_delivery      2159
Lý do phổ biến nhất:                  0
count  wrong_size


In [ ]:
# ------------------------------------------------------------
# 6) Q4
# ------------------------------------------------------------

q4_bounce_rate = web_traffic[["date", "traffic_source", "bounce_rate"]].dropna().copy()

q4_traffic_source_bounce_rate= q4_bounce_rate.groupby("traffic_source")["bounce_rate"].mean()

q4_best_traffic_source = q4_traffic_source_bounce_rate.idxmin()
q4_options={
    "organic_search": "A",
    "paid_search": "B", 
    "email_campaign": "C",
    "social_media": "D"
}
print("\n=== Q4 ===") 
print(q4_best_traffic_source)
q4_answer= q4_options.get(q4_best_traffic_source)
print(f"Closest option: {q4_answer} ({q4_best_traffic_source})")


=== Q4 ===
email_campaign
Closest option: C (email_campaign)


In [ ]:
# ------------------------------------------------------------
# 7) Q5
# ------------------------------------------------------------

total_order = len(order_items)
q5_order_with_discount= order_items["promo_id"].notnull().sum()
q5_ratio= (q5_order_with_discount/total_order)*100.
print(f"\n=== Q5 ===\nTỷ lệ áp dụng khuyến mãi: ",round(q5_ratio),"%")
q5_options={
    15 : "A" ,
    25 : "B" ,
    39 : "C" ,
    54 : "D"
}
q5_answer = min(q5_options, key=lambda x: abs(x - q5_ratio))
print("Closest option:", q5_options[q5_answer], f"({round(q5_ratio)} %)")



=== Q5 ===
Tỷ lệ áp dụng khuyến mãi:  39 %
Closest option: C (39 %)


In [10]:
# ---------------------------------------------------------------------------------
# 8) Q6
# ---------------------------------------------------------------------------------


order_counts = orders.groupby('customer_id').size().reset_index(name='order_count')
q6_data = customers.merge(order_counts, on='customer_id', how='left')
q6_data['order_count'] = q6_data['order_count'].fillna(0)
q6_final = q6_data[q6_data['age_group'].notnull()]
q6_result = q6_final.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print("Số đơn hàng trung bình theo nhóm tuổi:")
for age_group, avg_value in q6_result.items():
    print(f"{age_group:<10}: {avg_value:.4f}")

print("-" * 30)
print("Đáp án đúng là nhóm:", q6_result.idxmax())
q6_options={
    "55+" :"A",
    "25-34" :"B",   
    "35-44" :"C",
    "45-54" :"D"
}
q6_answer=q6_options.get(q6_result.idxmax())
print(f"Closest answer: {q6_answer} ({q6_result.idxmax()})")

Số đơn hàng trung bình theo nhóm tuổi:
55+       : 5.4069
45-54     : 5.3572
35-44     : 5.3373
25-34     : 5.2452
18-24     : 5.2267
------------------------------
Đáp án đúng là nhóm: 55+
Closest answer: A (55+)


In [ ]:
# ---------------------------------------------------------------------------------
# 09) Q7
# ---------------------------------------------------------------------------------

# 1. Gộp bảng đơn hàng (orders) với địa lý (geography) qua cột 'zip'
# Bước này để xác định vùng miền cho từng đơn hàng
orders_geo = pd.merge(orders, geography, on='zip', how='left')

# 2. Gộp tiếp với bảng thanh toán (payments) qua cột 'order_id'
# Bước này để lấy giá trị tiền thanh toán thực tế (payment_value)
combined_data = pd.merge(orders_geo, payments, on='order_id', how='left')

# 3. Tính tổng doanh thu theo từng vùng (region)
# Dùng ascending=False để vùng cao nhất nằm ở đầu danh sách
revenue_by_region = combined_data.groupby('region')['payment_value'].sum().sort_values(ascending=False)

# 4. IN KẾT QUẢ SẠCH (Loại bỏ Name và dtype)
print("Thống kê tổng doanh thu theo từng vùng:")
for region, total in revenue_by_region.items():
    # Kiểm tra nếu total là NaN thì in 0, ngược lại in số tiền định dạng 2 chữ số thập phân
    val = total if not pd.isna(total) else 0
    print(f"Vùng {region:<10}: {val:,.0f}")

print("-" * 40)
q7_options={
    'West':'A',
    'Central':'B',
    'East': 'C',
    ' Cả ba vùng có doanh thu xấp xỉ bằng nhau':'D'
}
q7_answer = q7_options.get(revenue_by_region.idxmax())
print(f"Closest answer: {q7_answer} ({revenue_by_region.idxmax()})")

Thống kê tổng doanh thu theo từng vùng:
Vùng East      : 7,291,150,819
Vùng Central   : 4,719,491,268
Vùng West      : 3,670,227,178
----------------------------------------
Closest answer: C (East)


In [16]:
# ---------------------------------------------------------------------------------
# 10) Q8
# ---------------------------------------------------------------------------------
q8_cancelled_order= orders[orders["order_status"]=="cancelled"]
q8_counts = q8_cancelled_order['payment_method'].value_counts()
q8_most_method = q8_counts.idxmax()
q8_options={
    "credit_card" : "A",
    "cod" : "B",
    "paypal" : "C",
    "bank_transfer" : "D"
}
q8_answer= q8_options.get(q8_most_method)
print("Thống kê thanh toán của các đơn hàng bị hủy:")
for method, count in q8_counts.items():
    print(f"{method:<20}: {count}")
print("-" * 30)
print(f"Closest answer: {q8_answer} ({q8_most_method})" )




Thống kê thanh toán của các đơn hàng bị hủy:
credit_card         : 28452
cod                 : 15468
paypal              : 7817
apple_pay           : 5190
bank_transfer       : 2535
------------------------------
Closest answer: A (credit_card)


In [ ]:
# ---------------------------------------------------------------------------------
# 11) Q9: Kích thước sản phẩm nào có tỷ lệ trả hàng cao nhất
# ---------------------------------------------------------------------------------

product_sizes = products[['product_id', 'size']]
ordered_with_size = order_items.merge(product_sizes, on='product_id', how='left')
ordered_counts = ordered_with_size.groupby('size').size()

returned_with_size = returns.merge(product_sizes, on='product_id', how='left')
returned_counts = returned_with_size.groupby('size').size()

target_sizes = ['S', 'M', 'L', 'XL']
q9_return_rates = (returned_counts.reindex(target_sizes) / ordered_counts.reindex(target_sizes))

print("Thống kê tỷ lệ trả hàng theo kích thước:")
for size, rate in q9_return_rates.items():
    val = rate if not pd.isna(rate) else 0
    print(f"Size {size:<3}: {val:.4f}")

print("-" * 30)
q9_options={
    'S' :'A',
    'M' :'B',
    'L' :'C',
    'XL':'D'
}
q9_answer=q9_options.get(q9_return_rates.idxmax())
print(f"Closest answer: {q9_answer} ({q9_return_rates.idxmax()})" )

Thống kê tỷ lệ trả hàng theo kích thước:
Size S  : 0.0565
Size M  : 0.0557
Size L  : 0.0562
Size XL : 0.0552
------------------------------
Closest answer: A (S)


In [ ]:
# ---------------------------------------------------------------------------------
# 12) Q10: Kế hoạch trả góp nào có giá trị thanh toán trung bình cao nhất
# ---------------------------------------------------------------------------------

q10_result = payments.groupby('installments')['payment_value'].mean()
target_installments = [1, 3, 6, 12]
q10_filtered = q10_result.reindex(target_installments).sort_values()

print("Giá trị thanh toán trung bình theo từng kỳ hạn:")
for installment, avg_value in q10_filtered.items():
    if not pd.isna(avg_value):
        print(f"Kỳ hạn {int(installment):>2} kỳ : {avg_value:.2f}")

print("-" * 35)

q10_options={
    1: 'A',
    3: 'B',
    6: 'C',
    12: 'D'
}

q10_answer =q10_options.get(q10_filtered.idxmax())
print(f"Closest answer: {(q10_answer)} ({q10_filtered.idxmax()} kỳ) ")

Giá trị thanh toán trung bình theo từng kỳ hạn:
Kỳ hạn  1 kỳ : 24113.27
Kỳ hạn 12 kỳ : 24245.77
Kỳ hạn  3 kỳ : 24399.64
Kỳ hạn  6 kỳ : 24446.65
-----------------------------------
Closest answer: C (6 kỳ) 
